# Teoria LLM gateway + LangChain

Este notebook reproduz os exemplos do README:

- **OpenAI-compatible** `POST /v1/chat/completions` (não-stream e stream) via `langchain_openai.ChatOpenAI`.
- **API simplificada** `POST /api/v1/chat` com `input` + `stream` via `httpx` (não há binding oficial LangChain para esse path).

## Variáveis de ambiente

| Variável | Significado | Padrão (teste local) |
|----------|-------------|----------------------|
| `TEORIA_BASE_URL` | Origem do gateway (sem barra final). Use `https://llm.jambu.ai` em produção ou `http://localhost:8081` com o stack `make test-up`. | `http://localhost:8081` |
| `TEORIA_API_KEY` | Chave (header `Authorization: Bearer …` ou equivalente). | `test-key-123` |
| `TEORIA_MODEL` | Nome do modelo enviado pelo cliente; o gateway pode sobrescrever com `VLLM_MODEL`. | `gpt-4o-mini` |

Instalação: `pip install -r notebooks/requirements.txt` (ou `uv pip install -r notebooks/requirements.txt`).


In [ ]:
import json
import os

import httpx
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

TEORIA_BASE = os.environ.get("TEORIA_BASE_URL", "http://localhost:8081").rstrip("/")
TEORIA_API_KEY = os.environ.get(
    "TEORIA_API_KEY", os.environ.get("GATEWAY_API_KEY", "test-key-123")
)
# LangChain sempre envia um campo model; no gateway ele é opcional e pode ser resolvido no servidor.
TEORIA_MODEL = os.environ.get("TEORIA_MODEL", "gpt-4o-mini")

OPENAI_COMPAT_BASE = f"{TEORIA_BASE}/v1"
SIMPLIFIED_CHAT_URL = f"{TEORIA_BASE}/api/v1/chat"

print("TEORIA_BASE:", TEORIA_BASE)
print("OpenAI-compatible:", f"{OPENAI_COMPAT_BASE}/chat/completions")
print("Simplified:", SIMPLIFIED_CHAT_URL)


## 1) OpenAI-compatible — conclusão única (README: `curl` sem `stream`)

Equivalente a `messages` + `max_tokens` em `/v1/chat/completions`.


In [ ]:
llm = ChatOpenAI(
    base_url=OPENAI_COMPAT_BASE,
    api_key=TEORIA_API_KEY,
    model=TEORIA_MODEL,
    max_tokens=512,
)

response = llm.invoke([HumanMessage(content="Hello")])
print(response.content)


## 2) OpenAI-compatible — streaming (`stream: true`)

LangChain usa o mesmo endpoint; o cliente HTTP negocia SSE como no `curl -N` do README.


In [ ]:
for chunk in llm.stream([HumanMessage(content="Hello")]):
    if chunk.content:
        print(chunk.content, end="", flush=True)
print()


## 3) API simplificada `/api/v1/chat` (README: `input` + `stream`)

**Compatibilidade LangChain:** `ChatOpenAI` fala apenas com rotas estilo OpenAI (`/v1/chat/completions`). O contrato `input` / `system_prompt` é outro path; aqui usamos `httpx`. A resposta **não-stream** do gateway ainda é JSON no formato OpenAI (`choices[0].message.content`), o que facilita o parse manual.


In [ ]:
headers = {
    "Authorization": f"Bearer {TEORIA_API_KEY}",
    "Content-Type": "application/json",
}

with httpx.Client(timeout=120.0) as http:
    r = http.post(
        SIMPLIFIED_CHAT_URL,
        json={
            "input": "What is 2+2? Reply with one word.",
            "max_tokens": 32,
            "stream": False,
        },
        headers=headers,
    )
    r.raise_for_status()
    body = r.json()
    print("non-stream:", body["choices"][0]["message"]["content"])

    with http.stream(
        "POST",
        SIMPLIFIED_CHAT_URL,
        json={
            "input": "Count from 1 to 10, one per line",
            "max_tokens": 100,
            "stream": True,
        },
        headers=headers,
    ) as resp:
        resp.raise_for_status()
        pieces: list[str] = []
        for line in resp.iter_lines():
            if not line or not line.startswith("data: "):
                continue
            payload = line[6:]
            if payload == "[DONE]":
                break
            chunk = json.loads(payload)
            delta = chunk["choices"][0].get("delta") or {}
            piece = delta.get("content") or ""
            if piece:
                pieces.append(piece)
                print(piece, end="", flush=True)
        print("\nstream chars:", sum(len(p) for p in pieces))


## Notas de compatibilidade (checklist)

1. **`model` obrigatório no cliente LangChain** — O README diz que o campo é opcional no gateway; o construtor `ChatOpenAI` exige um nome. Use qualquer string estável ou alinhe com o modelo real; se `VLLM_MODEL` estiver definido no gateway, ele pode substituir o valor enviado.
2. **`/api/v1/chat` fora do ecossistema OpenAI da LangChain** — Use `httpx` (ou um wrapper customizado / `RunnableLambda`) se precisar de `input` + `system_prompt` no path simplificado.
3. **Auth** — `Authorization: Bearer` funciona; em localhost o README também documenta `x-api-key`. Ambos são aceites pelo gateway.
4. **Streaming** — SSE segue o formato OpenAI (`data: {json}` e `data: [DONE]`); o parser acima assume isso (igual ao mock vLLM e ao backend real).
